# 03. Benchmark Comparativo: Modelo Atual vs Novo Modelo
Este notebook permite realizar um teste comparativo rigoroso e científico entre o **modelo atual** (treinado anteriormente) e o **novo modelo** (treinado com aumento dinâmico de rotação e duplicação fotométrica).

### Métricas Avaliadas:
1. **mAP50 e mAP50-95:** Precisão média nas caixas delimitadoras.
2. **Recall (Revocação):** Capacidade de não deixar nenhuma placa passar batida.
3. **Precisão ($P$):** Certeza de que o que foi detectado é realmente uma placa (redução de falsos positivos).
4. **Teste de Estresse com Rotação:** Desempenho específico em placas inclinadas (diagonais) e fotos noturnas.
5. **Tempo de Inferência (Latência em ms):** Velocidade de processamento por frame.

---
## 1. Instalação de Dependências e Configuração do Ambiente

In [ ]:
!pip install -q ultralytics matplotlib pandas tabulate opencv-python

import cv2
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

print(f"CUDA disponível? {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo GPU: {torch.cuda.get_device_name(0)}")

---
## 2. Carregamento dos Dois Modelos na Pasta Raiz
Basta arrastar seus dois arquivos `.pt` (o atual/antigo e o novo) para a **pasta raiz** (`/content/` no Colab, o painel de arquivos à esquerda).

> [!TIP]
> Você pode nomeá-los como `modelo_antigo.pt` e `modelo_novo.pt` (ou deixar qualquer nome `.pt`). O script detecta automaticamente os dois arquivos na raiz ou abre uma janela para selecioná-los.

In [ ]:
import os
from pathlib import Path
from ultralytics import YOLO

# =====================================================================
# CONFIGURAÇÃO DOS MODELOS NA PASTA RAIZ (/content/ no Colab)
# =====================================================================
# Nomes sugeridos para os modelos na pasta raiz:
NOME_MODELO_ATUAL = 'modelo_antigo.pt'  # ou o modelo treinado ontem
NOME_MODELO_NOVO  = 'modelo_novo.pt'    # ou o modelo com augmentation
# =====================================================================

pasta_raiz = Path('/content') if Path('/content').exists() else Path('.')

caminho_a = pasta_raiz / NOME_MODELO_ATUAL
caminho_b = pasta_raiz / NOME_MODELO_NOVO

# Busca todos os arquivos .pt na pasta raiz
arquivos_pt = sorted([f for f in pasta_raiz.glob('*.pt') if not f.name.startswith(('yolov8', 'yolo11'))])

# 1. Se os nomes configurados não existirem mas houver 2 modelos .pt na raiz, usa-os automaticamente
if not (caminho_a.exists() and caminho_b.exists()):
    if len(arquivos_pt) == 2:
        caminho_a, caminho_b = arquivos_pt[0], arquivos_pt[1]
        print(f'🔍 Detectados automaticamente 2 modelos na pasta raiz:')
        print(f'   - Modelo A: {caminho_a.name}')
        print(f'   - Modelo B: {caminho_b.name}')

# 2. Se ainda assim faltar algum modelo, abre o seletor de arquivos do Colab
if not (caminho_a.exists() and caminho_b.exists()):
    print('📂 Modelos não encontrados na raiz. Clique abaixo para enviar os dois arquivos (.pt):')
    try:
        from google.colab import files
        uploaded = files.upload()
        arquivos_pt = sorted([f for f in pasta_raiz.glob('*.pt') if not f.name.startswith(('yolov8', 'yolo11'))])
        if len(arquivos_pt) >= 2:
            caminho_a, caminho_b = arquivos_pt[0], arquivos_pt[1]
    except Exception as e:
        print(f'Nota: upload interativo indisponível ({e}).')

assert caminho_a.exists(), f'❌ Modelo A não encontrado em: {caminho_a}'
assert caminho_b.exists(), f'❌ Modelo B não encontrado em: {caminho_b}'

print('\n🚀 Carregando modelos...')
modelo_a = YOLO(str(caminho_a))
modelo_b = YOLO(str(caminho_b))

print(f'✅ Modelo A (Atual) pronto: {caminho_a.name}')
print(f'✅ Modelo B (Novo)  pronto: {caminho_b.name}')

---
## 3. Avaliação Formal no Conjunto de Teste Oficial
Executa a validação em lote no conjunto de teste independente (`split='test'`), extraindo as métricas científicas padronizadas pela Ultralytics/COCO.

In [ ]:
import io
import zipfile
import urllib.request
from pathlib import Path

# Verifica se algum dataset já está pronto no ambiente
YAML_DATASET = None
for cand in ['/content/placas_brasil.yaml', '/content/placas.yaml', 'placas_brasil.yaml', 'placas.yaml']:
    if Path(cand).exists():
        YAML_DATASET = cand
        break

# Se não houver dataset no ambiente, baixa e estrutura o dataset brasileiro automaticamente em segundos
if YAML_DATASET is None:
    print('📦 Dataset não detectado no ambiente. Baixando conjunto de teste brasileiro...')
    url = 'https://raw.githubusercontent.com/jhonathasmoura/Deteccao_Placas_Veiculos_YOLOv4/main/dataset.zip'
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as resp:
        zip_bytes = resp.read()
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        z.extractall('/content/dataset_brasil_raw')
    
    base_dest = Path('/content/dataset_brasil')
    for pasta_origem, split_dest in [('train', 'train'), ('valid', 'val'), ('test', 'test')]:
        img_dir = base_dest / 'images' / split_dest
        lbl_dir = base_dest / 'labels' / split_dest
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)
        origem = Path(f'/content/dataset_brasil_raw/{pasta_origem}')
        for arq in origem.glob('*.*'):
            if arq.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                arq.replace(img_dir / arq.name)
            elif arq.suffix.lower() == '.txt':
                arq.replace(lbl_dir / arq.name)
                
    yaml_content = '''
path: /content/dataset_brasil
train: images/train
val: images/val
test: images/test
names:
  0: placa
'''
    with open('/content/placas_brasil.yaml', 'w', encoding='utf-8') as f:
        f.write(yaml_content.strip())
    YAML_DATASET = '/content/placas_brasil.yaml'
    print('✅ Dataset brasileiro pronto para avaliação!')

print(f'📊 Avaliando no dataset: {YAML_DATASET}')
print('⏳ Avaliando Modelo A (Atual)... espere alguns segundos...')
res_a = modelo_a.val(data=YAML_DATASET, split='test', verbose=False)

print('⏳ Avaliando Modelo B (Novo)... espere alguns segundos...')
res_b = modelo_b.val(data=YAML_DATASET, split='test', verbose=False)

# Estruturação do Quadro Comparativo
metricas_comparativas = {
    'Métrica': [
        'Precisão (Precision)',
        'Revocação (Recall)',
        'mAP @ 0.50 (IoU 50%)',
        'mAP @ 0.50:0.95',
        'Tempo Inferência (ms/img)'
    ],
    'Modelo A (Atual)': [
        f"{res_a.box.mp:.3f}",
        f"{res_a.box.mr:.3f}",
        f"{res_a.box.map50:.3f}",
        f"{res_a.box.map:.3f}",
        f"{res_a.speed['inference']:.2f} ms"
    ],
    'Modelo B (Novo)': [
        f"{res_b.box.mp:.3f}",
        f"{res_b.box.mr:.3f}",
        f"{res_b.box.map50:.3f}",
        f"{res_b.box.map:.3f}",
        f"{res_b.speed['inference']:.2f} ms"
    ],
    'Evolução (Δ)': [
        f"{(res_b.box.mp - res_a.box.mp):+.3f}",
        f"{(res_b.box.mr - res_a.box.mr):+.3f}",
        f"{(res_b.box.map50 - res_a.box.map50):+.3f}",
        f"{(res_b.box.map - res_a.box.map):+.3f}",
        f"{(res_b.speed['inference'] - res_a.speed['inference']):+.2f} ms"
    ]
}

df_comparativo = pd.DataFrame(metricas_comparativas)
print('\n=================== QUADRO COMPARATIVO ===================')
display(df_comparativo)

# Gráfico de Barras Comparativo
labels = ['Precisão', 'Recall', 'mAP50', 'mAP50-95']
valores_a = [res_a.box.mp, res_a.box.mr, res_a.box.map50, res_a.box.map]
valores_b = [res_b.box.mp, res_b.box.mr, res_b.box.map50, res_b.box.map]

x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(10, 5))
plt.bar(x - width/2, valores_a, width, label=f'Modelo A ({Path(caminho_a).stem})', color='#718096')
plt.bar(x + width/2, valores_b, width, label=f'Modelo B ({Path(caminho_b).stem})', color='#3182CE')
plt.ylabel('Pontuação (0.0 a 1.0)')
plt.title('Comparativo de Desempenho Geral (Test Set)')
plt.xticks(x, labels)
plt.ylim(0, 1.05)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

---
## 4. Teste de Estresse: Placas Diagonais e Condições Adversas
Aqui testamos a capacidade de cada modelo em lidar com o problema exato que você enfrentou: **placas inclinadas e cenários noturnos**.
Rotacionamos imagens de teste em vários ângulos (ex: $-25^\circ$, $-15^\circ$, $+15^\circ$, $+25^\circ$) e medimos a **confiança média de detecção** de cada modelo.

In [ ]:
# Localiza a pasta de imagens de teste
pasta_teste = None
for cand in [
    Path('/content/dataset_brasil/images/test'),
    Path('/content/dataset/images/test'),
    Path('/content/dataset_brasil/images/val'),
    Path('/content/dataset/images/val'),
    Path('dataset_brasil/images/test'),
    Path('dataset/images/test'),
]:
    if cand.exists():
        pasta_teste = cand
        break

if pasta_teste is None:
    raise FileNotFoundError('Pasta de teste não encontrada. Verifique se o dataset foi estruturado na Seção 3.')

print(f'📁 Usando imagens de teste de: {pasta_teste}')
amostras = [
    f for f in pasta_teste.iterdir()
    if f.suffix.lower() in ['.jpg', '.jpeg', '.png']
][:8]

assert len(amostras) > 0, f'Nenhuma imagem encontrada em {pasta_teste}'

angulos_teste = [-25, -15, 0, 15, 25]

confiancas_a = {ang: [] for ang in angulos_teste}
confiancas_b = {ang: [] for ang in angulos_teste}

def rotacionar_imagem(img, angulo):
    h, w = img.shape[:2]
    centro = (w / 2.0, h / 2.0)
    M = cv2.getRotationMatrix2D(centro, angulo, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)

for img_path in amostras:
    img_original = cv2.imread(str(img_path))
    if img_original is None:
        continue
    for ang in angulos_teste:
        img_rot = rotacionar_imagem(img_original, ang) if ang != 0 else img_original
        
        # Inferência Modelo A
        pred_a = modelo_a(img_rot, verbose=False)[0]
        conf_a = float(pred_a.boxes.conf[0]) if len(pred_a.boxes) > 0 else 0.0
        confiancas_a[ang].append(conf_a)
        
        # Inferência Modelo B
        pred_b = modelo_b(img_rot, verbose=False)[0]
        conf_b = float(pred_b.boxes.conf[0]) if len(pred_b.boxes) > 0 else 0.0
        confiancas_b[ang].append(conf_b)

medias_a = [float(np.mean(confiancas_a[ang])) if confiancas_a[ang] else 0.0 for ang in angulos_teste]
medias_b = [float(np.mean(confiancas_b[ang])) if confiancas_b[ang] else 0.0 for ang in angulos_teste]

plt.figure(figsize=(9, 5))
plt.plot(angulos_teste, medias_a, 'o--', color='#E53E3E', label=f'Modelo A ({Path(caminho_a).stem})', linewidth=2)
plt.plot(angulos_teste, medias_b, 's-', color='#38A169', label=f'Modelo B ({Path(caminho_b).stem})', linewidth=2)
plt.title('Confiança Média de Detecção vs Ângulo de Inclinação da Placa')
plt.xlabel('Ângulo de Inclinação (Graus)')
plt.ylabel('Confiança Média (0.0 a 1.0)')
plt.ylim(-0.05, 1.05)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

---
## 5. Comparação Visual Lado a Lado (Side-by-Side)
Coloca os dois modelos frente a frente na mesma imagem desafiadora.

In [ ]:
# Se você tiver uma foto específica (como o carro preto noturno com placa inclinada):
# Envie a imagem pelo Colab ou selecione uma imagem do dataset
caminho_teste_desafio = str(amostras[0]) if amostras else None

# Você pode trocar para um arquivo local que tenha feito upload:
# caminho_teste_desafio = 'carro_inclinado.jpg'

if caminho_teste_desafio and os.path.exists(caminho_teste_desafio):
    img_bgr = cv2.imread(caminho_teste_desafio)
    # Simula uma inclinação de +20 graus para o duelo visual
    img_inclinada = rotacionar_imagem(img_bgr, 20)
    
    res_a = modelo_a(img_inclinada, verbose=False)[0]
    res_b = modelo_b(img_inclinada, verbose=False)[0]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))
    
    # Visualização A
    img_out_a = res_a.plot()
    ax1.imshow(cv2.cvtColor(img_out_a, cv2.COLOR_BGR2RGB))
    conf_a_txt = f"{res_a.boxes.conf[0]:.1%}" if len(res_a.boxes) > 0 else "Nenhuma"
    ax1.set_title(f"Modelo A (Atual)\nDetecção: {conf_a_txt}", color='red' if len(res_a.boxes) == 0 else 'black')
    ax1.axis('off')
    
    # Visualização B
    img_out_b = res_b.plot()
    ax2.imshow(cv2.cvtColor(img_out_b, cv2.COLOR_BGR2RGB))
    conf_b_txt = f"{res_b.boxes.conf[0]:.1%}" if len(res_b.boxes) > 0 else "Nenhuma"
    ax2.set_title(f"Modelo B (Novo com Augmentation)\nDetecção: {conf_b_txt}", color='green' if len(res_b.boxes) > 0 else 'black')
    ax2.axis('off')
    
    plt.tight_layout()
    plt.show()